# Gemma 4 context-pruning study
Kaggle: GPU T4 x2 (or P100), Internet ON, attach the `google/gemma-4` model (Transformers, `gemma-4-e4b-it`).
If no GPU or no model is available the notebook runs in DRY mode: it still builds every task, prunes context, measures token counts and checks whether the buggy line survives pruning, but skips generation (pass@1 and timing columns are then empty).

In [1]:
import re, json, time, subprocess, pathlib, random, ast, os, sys, tempfile, textwrap, shutil
BUDGETS=[1.0,0.75,0.5,0.3,0.15]
SEED=0; random.seed(SEED)
GEMMA_HANDLE=os.environ.get('GEMMA_HANDLE','google/gemma-4/transformers/gemma-4-e4b-it')
MAX_ATTEMPTS=int(os.environ.get('MAX_ATTEMPTS','1'))
MAX_NEW_TOKENS=512
FORCE_DRY=os.environ.get('DRY','0')=='1'

In [2]:
IDENT=re.compile(r'[A-Za-z_]\w+')
STRUCT=re.compile(r'\s*(def |class |import |from |@)')

def score_lines(lines, query_text, frame_lines=frozenset(), use_ids=True, use_frames=True, use_struct=True):
    ids=set(IDENT.findall(query_text))
    out=[]
    for i,l in enumerate(lines):
        s=0.0
        if use_ids: s+=len(ids & set(IDENT.findall(l)))
        if use_frames and frame_lines: s+=3/(1+min(abs(i-f) for f in frame_lines))
        if use_struct and STRUCT.match(l): s+=100
        out.append(s)
    return out

def enclosing_block(lines, i):
    # indices of the def/class block that contains line i
    j=i
    while j>=0 and not re.match(r'\s*(def |class )', lines[j]): j-=1
    if j<0: return {i}
    ind=len(lines[j])-len(lines[j].lstrip()); k=j+1
    while k<len(lines) and (not lines[k].strip() or len(lines[k])-len(lines[k].lstrip())>ind): k+=1
    return set(range(j,k))

def prune(text, query, k, frames=frozenset(), keep_frame_blocks=True, **kw):
    lines=text.splitlines()
    if k>=1.0: return text
    sc=score_lines(lines, query, frames, **kw)
    n=max(1,int(len(lines)*k))
    must=set()
    if keep_frame_blocks:
        for f in frames: must|=enclosing_block(lines,f)
    rest=[i for i in sorted(range(len(lines)), key=lambda i:-sc[i]) if i not in must]
    keep=must|set(rest[:max(0,n-len(must))])
    return '\n'.join(lines[i] for i in sorted(keep))

def random_prune(text, k, seed=0):
    lines=text.splitlines(); r=random.Random(seed)
    keep=sorted(r.sample(range(len(lines)), max(1,int(len(lines)*k))))
    return '\n'.join(lines[i] for i in keep)

def traceback_only(text, frames):
    lines=text.splitlines(); keep=set()
    for f in frames: keep|=enclosing_block(lines,f)
    return '\n'.join(lines[i] for i in sorted(keep))

## Tasks
A built-in benchmark of small single-function Python bugs, each embedded in a module padded with unrelated but realistic helper code so pruning has something to remove. Each task has a failing test. To use competition data instead, set `TASKS_JSON` to a JSON list of `{id, files:{path:src}, target, test, issue}`.

In [3]:
BUGS=[
 ('mean','def mean(xs):\n    return sum(xs)/(len(xs)+1)\n','assert mean([2,4])==3','mean() returns the wrong average'),
 ('clamp','def clamp(x, lo, hi):\n    return max(hi, min(lo, x))\n','assert clamp(5,0,10)==5 and clamp(-1,0,10)==0 and clamp(11,0,10)==10','clamp() does not keep values inside the range'),
 ('is_palindrome','def is_palindrome(s):\n    s=s.lower()\n    return s==s[1:][::-1]\n','assert is_palindrome("Abba") and not is_palindrome("abc")','is_palindrome() rejects real palindromes'),
 ('fib','def fib(n):\n    a,b=0,1\n    for _ in range(n-1):\n        a,b=b,a+b\n    return a\n','assert [fib(i) for i in range(1,7)]==[1,1,2,3,5,8]','fib() is off by one'),
 ('chunk','def chunk(xs, n):\n    return [xs[i:i+n] for i in range(0, len(xs)-n, n)]\n','assert chunk([1,2,3,4,5],2)==[[1,2],[3,4],[5]]','chunk() drops the tail'),
 ('word_count','def word_count(s):\n    return len(s.split(" "))\n','assert word_count("a  b c")==3','word_count() miscounts with mixed whitespace'),
 ('safe_div','def safe_div(a, b):\n    if b==0:\n        return 0\n    return a//b\n','assert safe_div(7,2)==3.5 and safe_div(1,0)==0','safe_div() truncates results'),
 ('dedupe','def dedupe(xs):\n    return list(set(xs))\n','assert dedupe([3,1,3,2,1])==[3,1,2]','dedupe() loses original order'),
 ('flatten','def flatten(xss):\n    out=[]\n    for xs in xss:\n        out.append(xs)\n    return out\n','assert flatten([[1],[2,3],[]])==[1,2,3]','flatten() returns nested lists'),
 ('celsius','def to_celsius(f):\n    return (f-32)*9/5\n','assert abs(to_celsius(212)-100)<1e-9','to_celsius() gives wrong temperatures'),
 ('last_index','def last_index(xs, v):\n    for i,x in enumerate(xs):\n        if x==v:\n            return i\n    return -1\n','assert last_index([1,2,1],1)==2 and last_index([],1)==-1','last_index() returns the first match'),
 ('median','def median(xs):\n    s=sorted(xs); n=len(s)\n    return s[n//2]\n','assert median([1,3,2])==2 and median([4,1,2,3])==2.5','median() is wrong for even-length input'),
]

def filler(seed, n=40):
    r=random.Random(seed); words=['parse','load','emit','scale','merge','index','cache','route','encode','score']
    out=[]
    for j in range(n):
        a,b=r.sample(words,2); name=f'{a}_{b}_{j}'
        out.append(f'def {name}(data, factor={r.randint(1,9)}):\n    """Helper {j}: {a} then {b}."""\n    acc=[]\n    for item in data:\n        if item is None:\n            continue\n        acc.append(item*factor)\n    return acc\n')
    return out

def make_tasks():
    tasks=[]
    for t,(name,src,test,issue) in enumerate(BUGS):
        fns=filler(t); pos=len(fns)//2
        module='import math\nimport itertools\n\n'+'\n'.join(fns[:pos]+[src]+fns[pos:])
        func=re.search(r'def (\w+)',src).group(1)
        tasks.append(dict(id=name, files={'lib.py':module}, target='lib.py', func=func,
            test=f'from lib import *\ndef test_{func}():\n    {test}\ntest_{func}()\n', issue=issue))
    return tasks

tasks=json.load(open(os.environ['TASKS_JSON'])) if os.environ.get('TASKS_JSON') else make_tasks()
print(len(tasks),'tasks')

12 tasks


In [4]:
def run_test(t, files=None):
    d=pathlib.Path(tempfile.mkdtemp())
    for p,s in (files or t['files']).items(): (d/p).write_text(s)
    (d/'test_task.py').write_text(t['test'])
    try: r=subprocess.run([sys.executable,'test_task.py'],cwd=d,capture_output=True,text=True,timeout=20)
    except subprocess.TimeoutExpired: shutil.rmtree(d); return False,'timeout'
    shutil.rmtree(d); return r.returncode==0, r.stderr

def frames_from_tb(tb, fname):
    return frozenset(int(m)-1 for m in re.findall(r'File "[^"]*'+re.escape(fname)+r'", line (\d+)', tb))

def bug_lines(t):
    lines=t['files'][t['target']].splitlines()
    i=next(i for i,l in enumerate(lines) if l.startswith('def '+t['func']+'('))
    return enclosing_block(lines,i)

for t in tasks:
    ok,tb=run_test(t); assert not ok, t['id']+' should fail before fixing'
    t['traceback']=tb[-1500:]
    # traceback usually points at the test, so also add frames for every lib.py line named
    t['frames']=frames_from_tb(tb,t['target']); t['tb_frames']=bool(t['frames'])
    if not t['frames']:
        # assertion failures stop in the test file, so fall back to defs of functions named in the traceback
        names=set(IDENT.findall(tb)); lines=t['files'][t['target']].splitlines()
        t['frames']=frozenset(i for i,l in enumerate(lines) if (m:=re.match(r'def (\w+)',l)) and m.group(1) in names)
print('tasks with lib.py frames in raw traceback:',sum(t['tb_frames'] for t in tasks),'| with frames after name fallback:',sum(bool(t['frames']) for t in tasks))

tasks with lib.py frames in raw traceback: 0 | with frames after name fallback: 12


In [5]:
# Load Gemma 4 if a GPU and the model are available, else DRY mode
model=tok=None
try:
    import torch
    HAS_GPU=torch.cuda.is_available()
except ImportError:
    HAS_GPU=False
MODE='dry'
if HAS_GPU and not FORCE_DRY:
    import kagglehub
    from transformers import AutoProcessor, AutoModelForCausalLM
    MODEL_PATH=kagglehub.model_download(GEMMA_HANDLE)
    tok=AutoProcessor.from_pretrained(MODEL_PATH)
    model=AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16, device_map='auto').eval()
    MODE='gemma'
print('MODE =',MODE)

def ntokens(s):
    if tok is not None:
        tk=getattr(tok,'tokenizer',tok); return len(tk(s).input_ids)
    return len(re.findall(r'\w+|[^\w\s]', s))  # proxy in DRY mode

def build_prompt(t, ctx):
    return (f"Fix the bug in `{t['target']}`.\nIssue: {t['issue']}\n\nFailing test:\n```python\n{t['test']}```\n"
            f"Traceback:\n```\n{t['traceback']}```\nRepository context (may be pruned):\n```python\n{ctx}\n```\n"
            f"Reply with only the full corrected definition of `{t['func']}` in one ```python block.")

def generate(model, tok, t, ctx):
    msgs=[{'role':'user','content':build_prompt(t,ctx)}]
    text=tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inp=tok(text=text, return_tensors='pt').to(model.device); n=inp['input_ids'].shape[-1]
    torch.cuda.synchronize(); t0=time.time()
    with torch.no_grad(): model(**inp)  # prefill-only timing
    torch.cuda.synchronize(); prefill=time.time()-t0
    torch.manual_seed(SEED)
    with torch.no_grad(): out=model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return tok.decode(out[0][n:], skip_special_tokens=True), prefill, n

def apply_patch(t, reply):
    m=re.search(r'```(?:python)?\n(.*?)```', reply, re.S); code=m.group(1) if m else reply
    try:
        new=next(n for n in ast.walk(ast.parse(code)) if isinstance(n,ast.FunctionDef) and n.name==t['func'])
    except Exception: return None
    src=t['files'][t['target']]; tree=ast.parse(src)
    old=next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name==t['func'])
    lines=src.splitlines()
    newsrc='\n'.join(lines[:old.lineno-1]+ast.get_source_segment(code,new).splitlines()+lines[old.end_lineno:])
    return {**t['files'], t['target']:newsrc}

def apply_and_test(t, reply):
    files=apply_patch(t, reply)
    if files is None: return False, True
    return run_test(t, files)[0], False

MODE = dry


In [6]:
# Harness self-check: oracle fixes must make every test pass through the same apply path the model uses
FIXES={'mean':'def mean(xs):\n    return sum(xs)/len(xs)','clamp':'def clamp(x, lo, hi):\n    return max(lo, min(hi, x))',
 'is_palindrome':'def is_palindrome(s):\n    s=s.lower()\n    return s==s[::-1]','fib':'def fib(n):\n    a,b=0,1\n    for _ in range(n):\n        a,b=b,a+b\n    return a',
 'chunk':'def chunk(xs, n):\n    return [xs[i:i+n] for i in range(0, len(xs), n)]','word_count':'def word_count(s):\n    return len(s.split())',
 'safe_div':'def safe_div(a, b):\n    if b==0:\n        return 0\n    return a/b','dedupe':'def dedupe(xs):\n    return list(dict.fromkeys(xs))',
 'flatten':'def flatten(xss):\n    out=[]\n    for xs in xss:\n        out.extend(xs)\n    return out','to_celsius':'def to_celsius(f):\n    return (f-32)*5/9',
 'last_index':'def last_index(xs, v):\n    for i in range(len(xs)-1,-1,-1):\n        if xs[i]==v:\n            return i\n    return -1',
 'median':'def median(xs):\n    s=sorted(xs); n=len(s)\n    return s[n//2] if n%2 else (s[n//2-1]+s[n//2])/2'}
if not os.environ.get('TASKS_JSON'):
    chk=[apply_and_test(t,'```python\n'+FIXES[t['func']]+'\n```')[0] for t in tasks]
    assert all(chk), chk
    print('oracle fixes pass',sum(chk),'/',len(chk))

oracle fixes pass 12 / 12


In [7]:
def variants(t):
    src=t['files'][t['target']]; q=t['issue']+'\n'+t['test']+'\n'+t['traceback']; fr=t['frames']
    for k in BUDGETS: yield f'query {int(k*100)}%', k, prune(src,q,k,fr)
    yield 'random 30%', .3, random_prune(src,.3,SEED)
    yield 'no-ids 30%', .3, prune(src,q,.3,fr,use_ids=False)
    yield 'no-struct 30%', .3, prune(src,q,.3,fr,use_struct=False)
    yield 'no-frames 30%', .3, prune(src,q,.3,fr,use_frames=False,keep_frame_blocks=False)
    if fr: yield 'traceback-only', None, traceback_only(src,fr)

results=[]
for t in tasks:
    full=t['files'][t['target']]; full_tok=ntokens(full)
    bug=[full.splitlines()[i] for i in sorted(bug_lines(t)) if full.splitlines()[i].strip()]
    for name,k,ctx in variants(t):
        ctxl=set(ctx.splitlines())
        row=dict(task=t['id'],variant=name,k=k,tokens=ntokens(ctx),frac_tokens=ntokens(ctx)/full_tok,
                 bug_fn_kept=all(l in ctxl for l in bug))
        try: ast.parse(ctx); row['parses']=True
        except SyntaxError: row['parses']=False
        if MODE=='gemma':
            ok=False; fails=0; t0=time.time()
            for a in range(MAX_ATTEMPTS):
                reply,pre,n=generate(model,tok,t,ctx); ok,af=apply_and_test(t,reply); fails+=af
                if ok: break
            row.update(ok=ok,sec=time.time()-t0,prefill_sec=pre,prompt_tokens=n,apply_failures=fails,reply=reply[:2000])
        results.append(row)
json.dump(results, open('results.json','w'), indent=1)
print(len(results),'rows ->results.json')

120 rows ->results.json


In [8]:
import pandas as pd
df=pd.DataFrame(results)
agg=dict(frac_tokens=('frac_tokens','mean'),bug_fn_kept=('bug_fn_kept','mean'),parses=('parses','mean'))
if 'ok' in df: agg.update(pass1=('ok','mean'),prefill_sec=('prefill_sec','mean'),sec=('sec','mean'),apply_failures=('apply_failures','sum'))
summary=df.groupby('variant',sort=False).agg(**agg).round(3)
summary.to_csv('summary.csv'); summary

,frac_tokens,bug_fn_kept,parses
variant,,,
query 100%,1.000,1.000,1.0
query 75%,0.777,1.000,0.0
query 50%,0.558,1.000,0.0
query 30%,0.382,1.000,0.0
query 15%,0.239,1.000,0.0
random 30%,0.302,0.000,0.0
no-ids 30%,0.374,1.000,0.0
no-struct 30%,0.379,1.000,0.0
no-frames 30%,0.379,0.083,0.0
